# Ejercicios – Estadística Descriptiva y Análisis Univariado

Resuelve los siguientes ejercicios. **No mires las soluciones** hasta haber intentado cada uno.

Utiliza las librerías `numpy`, `pandas`, `matplotlib`, `seaborn` y `scipy.stats`.

---


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(123)

# Dataset de trabajo (simulado)
n = 400
df = pd.DataFrame({
    "ingresos": np.random.lognormal(mean=3.2, sigma=0.7, size=n),
    "edad": np.random.normal(loc=38, scale=12, size=n).clip(18, 75),
    "satisfaccion": np.random.randint(1, 6, size=n),          # 1-5
    "region": np.random.choice(["Norte", "Sur", "Este", "Oeste"], size=n, p=[0.35, 0.25, 0.25, 0.15]),
    "compra": np.random.binomial(1, 0.4, size=n)
})
# Introducimos algunos outliers artificiales en ingresos
df.loc[np.random.choice(df.index, 8, replace=False), "ingresos"] *= 4

df.head()


## Ejercicio 1 – Medidas de tendencia central y dispersión

1. Calcula media, mediana, desviación estándar e IQR de la variable `ingresos`.
2. Calcula las mismas medidas para `edad`.
3. Compara el sesgo (skewness) de ambas variables. ¿Cuál está más sesgada?
4. Interpreta brevemente por qué la media y la mediana difieren más en una de las dos variables.


In [2]:
# 1. Medidas para 'ingresos'
media_ing = df['ingresos'].mean()
mediana_ing = df['ingresos'].median()
std_ing = df['ingresos'].std()
iqr_ing = df['ingresos'].quantile(0.75) - df['ingresos'].quantile(0.25)
skew_ing = df['ingresos'].skew()

print('=== Variable: ingresos ===')
print(f'Media:               {media_ing:.2f}')
print(f'Mediana:             {mediana_ing:.2f}')
print(f'Desviación estándar: {std_ing:.2f}')
print(f'IQR:                 {iqr_ing:.2f}')
print(f'Sesgo (Skewness):    {skew_ing:.2f}\n')

# 2. Medidas para 'edad'
media_edad = df['edad'].mean()
mediana_edad = df['edad'].median()
std_edad = df['edad'].std()
iqr_edad = df['edad'].quantile(0.75) - df['edad'].quantile(0.25)
skew_edad = df['edad'].skew()

print('=== Variable: edad ===')
print(f'Media:               {media_edad:.2f}')
print(f'Mediana:             {mediana_edad:.2f}')
print(f'Desviación estándar: {std_edad:.2f}')
print(f'IQR:                 {iqr_edad:.2f}')
print(f'Sesgo (Skewness):    {skew_edad:.2f}')


**Respuesta e interpretación (Puntos 3 y 4):**
- **Comparación de sesgo:** La variable `ingresos` está mucho más sesgada (asimetría positiva alta), mientras que `edad` tiene un sesgo cercano a 0 (distribución prácticamente simétrica).
- **¿Por qué difieren la media y la mediana en `ingresos`?** Porque la media es muy sensible a los valores extremos (*outliers*), que tiran el promedio hacia arriba. En cambio, la mediana divide los datos al 50% y es robusta frente a esos valores altos. En `edad`, al tener una distribución simétrica y sin valores extremos fuertes, la media y la mediana son casi iguales.

## Ejercicio 2 – Visualización univariada

1. Construye un histograma con KDE de `ingresos` y otro de `edad`.
2. Construye un boxplot de `ingresos`.
3. Comenta visualmente la presencia de outliers y la forma de cada distribución.


In [3]:
# 1. Histogramas con KDE de 'ingresos' y 'edad'
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['ingresos'], kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Histograma y KDE de Ingresos')
axes[0].set_xlabel('Ingresos')

sns.histplot(df['edad'], kde=True, color='mediumseagreen', ax=axes[1])
axes[1].set_title('Histograma y KDE de Edad')
axes[1].set_xlabel('Edad')

plt.tight_layout()
plt.show()

# 2. Boxplot de 'ingresos'
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['ingresos'], color='salmon')
plt.title('Boxplot de Ingresos')
plt.xlabel('Ingresos')
plt.show()


**Comentario visual (Punto 3):**
- **`ingresos`:** Se observa una distribución fuertemente asimétrica positiva (concentrada a la izquierda con una cola muy larga hacia la derecha). En el Boxplot se ven numerosos puntos fuera del bigote derecho, que corresponden a los *outliers* de altos ingresos.
- **`edad`:** Tiene forma de campana simétrica (distribución normal) centrada alrededor de los 38 años, sin colas largas ni valores atípicos extremos.

## Ejercicio 3 – Variables categóricas

1. Construye la tabla de frecuencias absolutas y relativas de `region`.
2. Genera un countplot de `region` ordenado de mayor a menor frecuencia.
3. Si existiera una categoría con menos del 5 % de observaciones, ¿cómo la tratarías? Justifica.


In [4]:
# 1. Tabla de frecuencias absolutas y relativas de 'region'
frecuencia_abs = df['region'].value_counts()
frecuencia_rel = (df['region'].value_counts(normalize=True) * 100).round(2)

tabla_region = pd.DataFrame({
    'Frecuencia Absoluta': frecuencia_abs,
    'Frecuencia Relativa (%)': frecuencia_rel
})
print(tabla_region)

# 2. Countplot de 'region' ordenado de mayor a menor frecuencia
orden_regiones = df['region'].value_counts().index

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='region', order=orden_regiones, palette='viridis')
plt.title('Distribución de Observaciones por Región')
plt.xlabel('Región')
plt.ylabel('Frecuencia')
plt.show()


**Respuesta (Punto 3):**
Si existieran categorías raras con menos del 5 % de las observaciones, las agruparía dentro de una nueva categoría llamada **`"Otros"`** (o las combinaría con una región vecina/similar si tiene sentido de negocio). Esto simplifica la lectura de los gráficos y evita que un futuro modelo de Machine Learning se sobreajuste (*overfitting*) a categorías con muy pocos datos.

## Ejercicio 4 – Detección de outliers

1. Implementa una función que detecte outliers de `ingresos` usando el método IQR (factor 1.5).
2. Implementa otra función que use el Z-score con umbral 3.
3. Compara cuántos outliers detecta cada método.
4. ¿Cuál método crees más adecuado para esta variable? Justifica.


In [5]:
# 1. Función para detectar outliers con método IQR (factor 1.5)
def detectar_outliers_iqr(serie, factor=1.5):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - factor * iqr
    limite_sup = q3 + factor * iqr
    return serie[(serie < limite_inf) | (serie > limite_sup)]

# 2. Función para detectar outliers con Z-score (umbral 3)
def detectar_outliers_zscore(serie, umbral=3):
    z_scores = np.abs(stats.zscore(serie))
    return serie[z_scores > umbral]

# 3. Comparación de cantidad de outliers detectados
out_iqr = detectar_outliers_iqr(df['ingresos'])
out_z = detectar_outliers_zscore(df['ingresos'])

print(f'Outliers detectados con método IQR:     {len(out_iqr)}')
print(f'Outliers detectados con método Z-score: {len(out_z)}')


**Respuesta (Punto 4):**
El método **IQR es el más adecuado** para la variable `ingresos`. Como `ingresos` no sigue una distribución normal sino que tiene un fuerte sesgo positivo y valores extremos, la media y la desviación estándar que usa el **Z-score** se ven distorsionadas (infladas) por los propios *outliers*, haciendo que detecte menos atípicos de los que realmente hay. En cambio, el **IQR** se basa en cuartiles ($Q_1$ y $Q_3$), por lo que es robusto frente a distribuciones asimétricas.

## Ejercicio 5 – Transformaciones

1. Aplica la transformación logarítmica a `ingresos`.
2. Aplica Box-Cox a `ingresos` y reporta el valor de λ.
3. Compara el sesgo antes y después de cada transformación.
4. Visualiza los tres histogramas (original, log, Box-Cox) lado a lado.


In [6]:
# 1. Transformación logarítmica
ingresos_log = np.log(df['ingresos'])

# 2. Transformación Box-Cox y valor de lambda
ingresos_boxcox, lambda_optimo = stats.boxcox(df['ingresos'])
ingresos_boxcox = pd.Series(ingresos_boxcox)

print(f'Valor óptimo de lambda (Box-Cox): {lambda_optimo:.4f}\n')

# 3. Comparación del sesgo (skewness) antes y después
print('=== Comparación de Sesgo (Skewness) ===')
print(f'Original: {df["ingresos"].skew():.4f}')
print(f'Log:      {ingresos_log.skew():.4f}')
print(f'Box-Cox:  {ingresos_boxcox.skew():.4f}')

# 4. Visualización de los 3 histogramas lado a lado
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(df['ingresos'], kde=True, color='salmon', ax=axes[0])
axes[0].set_title(f'Original (Skew = {df["ingresos"].skew():.2f})')

sns.histplot(ingresos_log, kde=True, color='steelblue', ax=axes[1])
axes[1].set_title(f'Transformación Log (Skew = {ingresos_log.skew():.2f})')

sns.histplot(ingresos_boxcox, kde=True, color='mediumseagreen', ax=axes[2])
axes[2].set_title(f'Box-Cox (λ={lambda_optimo:.2f}, Skew = {ingresos_boxcox.skew():.2f})')

plt.tight_layout()
plt.show()


## Ejercicio 6 – Prueba de hipótesis simple

1. Divide `edad` en dos grupos: menores de 40 años y de 40 o más.
2. Realiza un t-test de Welch para comparar las medias de `ingresos` entre ambos grupos.
3. Realiza un Mann-Whitney U sobre las mismas muestras.
4. Reporta los p-valores e indica si rechazarías H₀ al nivel α = 0.05 en cada caso.


In [7]:
# 1. Dividimos en dos grupos según la edad
ingresos_menores_40 = df[df['edad'] < 40]['ingresos']
ingresos_mayores_40 = df[df['edad'] >= 40]['ingresos']

# 2. t-test de Welch (equal_var=False)
stat_t, p_val_t = stats.ttest_ind(ingresos_menores_40, ingresos_mayores_40, equal_var=False)

# 3. Prueba no paramétrica de Mann-Whitney U
stat_mw, p_val_mw = stats.mannwhitneyu(ingresos_menores_40, ingresos_mayores_40)

# 4. Reporte de resultados
print(f't-test de Welch -> Estadístico: {stat_t:.4f} | p-valor: {p_val_t:.4f}')
print(f'Mann-Whitney U  -> Estadístico: {stat_mw:.4f} | p-valor: {p_val_mw:.4f}')


**Interpretación (Punto 4):**
- En ambos tests (t-test de Welch y Mann-Whitney U), el **p-valor es mayor a $\alpha = 0.05$**.
- Por lo tanto, **no se rechaza la hipótesis nula ($H_0$)** en ningún caso: no hay evidencia estadística de que los ingresos difieran significativamente entre los menores de 40 años y los de 40 años o más. Además, dado que `ingresos` tiene fuerte asimetría y *outliers*, la prueba no paramétrica de **Mann-Whitney U** es la más confiable para esta comparación.

## Ejercicio 7 – Correlación

1. Calcula los coeficientes de Pearson, Spearman y Kendall entre `edad` e `ingresos`.
2. Genera un scatterplot con recta de regresión.
3. Interpreta los resultados: ¿hay evidencia de asociación lineal o monotónica?


In [8]:
# 1. Coeficientes de correlación de Pearson, Spearman y Kendall
r_pearson, p_pearson = stats.pearsonr(df['edad'], df['ingresos'])
r_spearman, p_spearman = stats.spearmanr(df['edad'], df['ingresos'])
r_kendall, p_kendall = stats.kendalltau(df['edad'], df['ingresos'])

print(f'Pearson:  r = {r_pearson:.4f} (p-valor = {p_pearson:.4f})')
print(f'Spearman: ρ = {r_spearman:.4f} (p-valor = {p_spearman:.4f})')
print(f'Kendall:  τ = {r_kendall:.4f} (p-valor = {p_kendall:.4f})')

# 2. Scatterplot con recta de regresión
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df, x='edad', y='ingresos',
    scatter_kws={'alpha': 0.5, 'color': 'steelblue'},
    line_kws={'color': 'crimson'}
)
plt.title('Relación entre Edad e Ingresos (Scatterplot + Regresión)')
plt.xlabel('Edad')
plt.ylabel('Ingresos')
plt.show()


**Interpretación (Punto 3):**
- Los tres coeficientes (**Pearson**, **Spearman** y **Kendall**) dan valores muy cercanos a **0** y sus p-valores son mayores a `0.05`.
- En el gráfico de dispersión se observa una nube de puntos sin tendencia y una recta de regresión prácticamente horizontal. Por lo tanto, **no existe evidencia de asociación lineal ni monotónica** entre la edad y los ingresos en este dataset.

---
**Fin de los ejercicios.**  
Entrega este notebook con tus respuestas y código ejecutado.
